# **Goal**

The purpose of these notebooks is to present some initial experiments to motivate model selection for an interactive app that allows users to experiment with small-medium language model (100M - 1B range).

####**Qwen3-1.7B**

**This is the post-trained version of Qwen fine tuned for conversation**

- <u>Total params</u>: 1.7B. This is the total number of learned scalar values in the entire model: weights, biases, embeddings, output projection, etc.
- <u>Non-embdedding params</u>: 1.4B. These are the learned parameters in the transformer machinery itself, excluding the token embedding and output vocabulary matrices.
- <u>Number of layers</u>: 28
- <u>Model dimension</u>: 2048. I.e.: Every token is represented internally by a vector of length 2048.
- <u>MLP units per layer:</u> 6144
- <u>Attention Heads</u>: 16 for Q and 8 for KV
- <u>Vocabulary size:</u> 151,936. Amount of tokens in vocabulary.
- <u>Max context window</u>: 32,768 tokens.

Load model

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B", device_map="auto")
messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

<think>
Okay, the user is asking, "Who are you?" I need to answer that. Let me start by introducing myself.

First, I should mention my name. I'm called Qwen


In [2]:
print(model.config)

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 6144,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 40960,
  "max_w

Prompt experiments. Can play with the following generation parameters:

- `max_new_tokens:` Intenger determining length of response excluding initial prompt.
- `do_sample:` Boolean determining greedy/non-greedy sampling.
- `temperature:` Low temp -> sharp distribution; high temp -> flattened distribution. I.e., how predictable or conservative the model is when sampling tokens.
- `top_k:` Only consider the \(k\) highest-scoring tokens. Default = 50.
- `top_p:` Float 0-1.0. Keep the smallest set of most likely tokens whose probabilities together account for at least \(p\) of the probability mass. Default = 1.
- `repetition_penalty:` How much repeated tokens are discouraged. Default = 1.0 means no penalty.

In [3]:
messages = [
    {"role": "user", "content": "How much do you like ice cream?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=75)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

<think>
Okay, the user asked, "How much do you like ice cream?" Hmm, I need to figure out how to respond. Let me start by thinking about my role here. I'm an AI assistant, so I don't have personal preferences. But I should make it clear that I don't have feelings or tastes.

Wait, maybe the user is trying to


In [4]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

tokens = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(tokens[0]))

<|im_start|>user
The capital of France is<|im_end|>
<|im_start|>assistant
<think>
Okay, the user is asking for the capital of France. Let me think. I know that France is a country in Europe, and I recall that Paris is the capital. But I should double-check to make sure I'm not mixing it


In [5]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
)

print(tokenizer.decode(output[0]))

<|im_start|>user
The capital of France is<|im_end|>
<|im_start|>assistant
<think>
Okay, the user is asking for the capital of France. I know that France's capital is Paris. Let me make sure I get that right. Paris is the largest city in France and is known for the Eiffel Tower, the


In [6]:
messages = [
    {"role": "user", "content": "What is love?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=False,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(tokenizer.decode(output[0], skip_special_tokens=True))

Input tokens: tensor([151644,    872,    198,   3838,    374,   2948,     30, 151645,    198,
        151644,  77091,    198, 151667,    271, 151668,    271])
Output tokens: tensor([28251,   374,   264,  6351,   323, 61287,   580, 24481, 19772,   429,
        34227,  3941, 26735,    11,  7775,    11,   323, 37597,    13,  1084,
          646,   387,  4512,   304,  1657,  5510, 11649,   389,   279, 13057,
         1447,    16,    13,  3070,  2269, 40864, 11032, 95518, 10689,  3545,
        17601, 15650,   315, 19984,    11,  2453,    11,   323, 36705,   369])

user
What is love?
assistant
<think>

</think>

Love is a complex and multifaceted emotion that varies across cultures, individuals, and contexts. It can be defined in many ways depending on the perspective:

1. **Emotional Connection**: Love often involves feelings of attachment, care, and affection for


Here we are just printing the internal shape of one layer of the model.


In [7]:
print(model.model.layers[0].mlp)

Qwen3MLP(
  (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
  (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
  (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
  (act_fn): SiLUActivation()
)


In [8]:
model.config.intermediate_size

6144

The input vector representation is 1024-dimensional. The intermediate dimension is 3072. Note that the Qwen models use more sophisticated gated MLPS.

There are 2 separate sets of 3072 MLP units that receive the input. Each of these produces a a transformed and expanded 3072-d vector called the gate projection and the up projection. The gate projection goes through activation using SiLU (Sigmoid Linear Unit). Then this activated projection is multiplied element-wise with the up-projection, and these new values are transformed back into a 1024-d representation by another learned matrix. This representation will continue through the transformer block and subsequent layers.

Note: $\text{SiLU}(x) = x ⋅ σ(x)$, where $σ(x)$ is the sigmoid function $\frac{1}{1+e^{-x}}$.

**MLP Unit Ablation Experiments**

In [9]:
import torch

def add_mlp_ablation(model, layer_idx, proportion, seed=42):

    # MLP in layer layer_idx
    mlp = model.model.layers[layer_idx].mlp

    num_units = model.config.intermediate_size
    num_ablate = int(proportion * num_units)

    # Seed RNG
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Pick 'num_ablate' indexes for MLP units at random
    indices = torch.randperm(
        num_units,
        generator=generator
    )[:num_ablate]

    def ablation_hook(module, inputs):
        x = inputs[0].clone()
        indices_device = indices.to(x.device)
        x[..., indices_device] = 0
        return (x,) + inputs[1:]

    handle = mlp.down_proj.register_forward_pre_hook(ablation_hook)

    return handle, indices, layer_idx, proportion

Here we try zeroing the outputs of different proportions of the first (0th) layer. Notice gradual degradation in output coherence.

Baseline

In [10]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=False,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed activations in {proportion}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Input tokens: tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198, 151667,    271, 151668,    271])
Output tokens: tensor([   785,   6722,    315,   9625,    374,   3070,  59604,    334,     13,
        151645])

Zeroed activations in 0.0% of units in layer 0
user
The capital of France is
assistant
<think>

</think>

The capital of France is **Paris**.


$25\%$ of first layer

In [11]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.25,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Outputs: tensor([151667,    198,  32313,     11,    279,   1196,   4588,     11,    330,
           785,   6722,    315,   9625,    374,  21215,    358,   1184,    311,
          3410,    279,   4396,   4226,     13,   6771,    752,   1744,    382,
          5338,     11,    358,   6099,    429,   9625,    594,   6722,    374,
         12095,     13,   1988,   3783,     11,   7196,    807,   1366,    803,
          3565,     30,   8909,   3170,    432])

Zeroed activations in 25.0% of units in layer 0
user
The capital of France is
assistant
<think>
Okay, the user asked, "The capital of France is..." I need to provide the correct answer. Let me think.

First, I remember that France's capital is Paris. But wait, maybe they want more details? Like why it


$50\%$ of first layer

In [12]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.5,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Outputs: tensor([151667,    198,  32313,     11,    279,   1196,   4588,     11,    330,
           785,   6722,    315,   9625,    374,  21215,    358,   1184,    311,
          4226,    429,     13,   6771,    752,   1744,    382,   5338,     11,
           358,   1414,    429,  12095,    374,    279,   6722,   3283,    315,
          9625,     13,   1988,   3783,     11,   7196,    807,   1366,    803,
          3565,     30,   8909,   3170,    432])

Zeroed activations in 50.0% of units in layer 0
user
The capital of France is
assistant
<think>
Okay, the user asked, "The capital of France is..." I need to answer that. Let me think.

First, I know that Paris is the capital city of France. But wait, maybe they want more details? Like why it


$75\%$ of first layer

In [13]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.75,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Outputs: tensor([151667,    198,  32313,     11,    279,   1196,   4588,     11,    330,
           785,   6722,    315,   9625,    374,  21215,    358,   1184,    311,
          4226,    429,     13,   6771,    752,   1744,    382,   5338,     11,
           358,   1414,    429,  12095,    374,    279,   6722,   3283,    315,
          9625,     13,   1988,   3783,     11,   7196,    807,   1366,    803,
          3565,     30,   8909,   3170,    432])

Zeroed activations in 75.0% of units in layer 0
user
The capital of France is
assistant
<think>
Okay, the user asked, "The capital of France is..." I need to answer that. Let me think.

First, I know that Paris is the capital city of France. But wait, maybe they want more details? Like why it


$100\%$ of first layer

In [14]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=1.0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Outputs: tensor([151668,    271,  32313,     11,   1077,    594,   1490,     13,    576,
          1196,   6139,    330,    785,   6722,    315,   9625,    374,      1,
           323,   1221,   1045,   1008,   1467,     13,    358,   1184,    311,
          7071,    700,   1128,    807,   1366,    382,   4416,   1156,     11,
          7196,    279,   1196,   6801,    311,   1414,    279,   6722,   3283,
           315,   9625,     30,   2938,    594])

Zeroed activations in 100.0% of units in layer 0
user
The capital of France is
assistant
</think>

Okay, let's see. The user wrote "The capital of France is" and then some other text. I need to figure out what they want.

So first, maybe the user wants to know the capital city of France? That's


**Attention Head Ablation Experiments**

First let's check the attention heads

In [15]:
print(model.model.layers[0])

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
    (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
    (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
)


We see that the attention mechanism expects a 1024d input (the token embedding or output from the previous block). It then computes a 2048-d vector q_proj, and two 1024-d vectors, k_proj and v_proj. This corresponds to calculating its query, key, and value vectors. Note that there are are 16 Q-heads and 8-KV heads, so each key/value head is shared by two query heads. After computing attention scores using these 3 vectors, the attention mechanism produces 16 query-head outputs, each with dimension 128. This 2048 output goes into the o-projection, a learned projection from the concatenated attention head contributions, to the final attention contribution with dimension 1024.

In [16]:
print(model.model.layers[0].self_attn.head_dim)

128


In [17]:
model.config.num_hidden_layers

28

In [18]:
model.config.hidden_size

2048

In [19]:
model.config.num_attention_heads

16

Attention head ablation function

In [20]:
def add_head_ablation(model, head_indices, layer_idx):

    #Number of hidden layers
    num_layers = model.config.num_hidden_layers
    #Number of Query attention heads
    num_heads = model.config.num_attention_heads

    if not 0 <= layer_idx < num_layers:
        raise ValueError(f"layer_idx must be between 0 and {num_layers - 1}")
    for head_idx in head_indices:
        if not 0 <= head_idx < num_heads:
            raise ValueError(f"head_idx must be between 0 and {num_heads - 1}")

    #Projection dimension per attention head
    head_dim = model.model.layers[layer_idx].self_attn.head_dim

    # Load final attention module
    attention_dense = model.model.layers[layer_idx].self_attn.o_proj

    def head_ablation_hook(module, inputs):
        x = inputs[0].clone()

        for head_idx in head_indices:
            #compute indices corresponding to specific head
            start = head_idx * head_dim
            end = start + head_dim

            #Select indices in the head outputs vector corresponding to chosen head and zero them
            x[..., start:end] = 0

        return (x,) + inputs[1:]

    #Note this is a pre-hook because we want to intervene components before the o-projection forward pass
    handle = attention_dense.register_forward_pre_hook(head_ablation_hook)

    return handle, head_indices, layer_idx

Here we try zeroing the outputs of different numbers of heads of the first (0th) layer.

Remove 1 head from layer 0

In [21]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Output tokens: tensor([151667,    198,  32313,     11,    279,   1196,   4588,    369,    279,
          6722,    315,   9625,     13,   6771,    752,   1744,   1112,    358,
          1414,    429,  12095,    374,    279,   6722,     13,   1988,   3783,
            11,   7196,    807,   1366,    803,   3565,     30,   8909,   3170,
           432,    594,   2989,    476,   1045,  13656,   2266,   1939,  80022,
            11,   1265,    358,   6286,   1008])

Zeroed head [0] in layer 0
user
The capital of France is
assistant
<think>
Okay, the user asked for the capital of France. Let me think... I know that Paris is the capital. But wait, maybe they want more details? Like why it's important or some historical context?

Hmm, should I mention other


Ablate 4 attention heads in layer 0

---



In [22]:
to_ablate = int(model.config.num_attention_heads*0.25)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Outputs: tensor([151667,    198,    510,    198,    198,    198,    279,    304,    198,
           198,    198,    198,    198,    198,    198,    198,    198,    198,
           198,    198,    198,    311,    271,  32313,     11,     13,    271,
           198,    389,   1008,   1091,    429,    323,    271,    358,   2776,
         14589,    911,    429,   1112,  13824,     11,   2146,     40,   1184,
           311,   1492,    448,    419,   3491])

Zeroed head [0, 1, 2, 3] in layer 0
user
The capital of France is
assistant
<think>
:



 the in












 to

Okay,.


 on other than that and

 I'm sorry about that... Wait,...

I need to help with this problem


Ablate 8 attention heads in layer 0

In [23]:
to_ablate = int(model.config.num_attention_heads*0.5)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198])
Outputs: tensor([  271,  2132,  4977,  3784,   220,   279,    11,    13,   304,   382,
         1986,  3491,   702,   624,  2610,   525,   614,   498,   311,   258,
         1055,    64,  1782,   323,  2146,    40,  2776,   537,  2704,  1112,
          358,  1079,  9338, 32313,     0,  6771,   594,   752,  1414,   330,
          264,     1,   263,    66,   389, 20225, 10061,   432,   387,  1934])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7] in layer 0
user
The capital of France is


It seems à  the,. in.

This problem has.
You are have you toinofathe and...

I'm not sure... I am...
Okay! Let's me know " a"onc on....

Let it be....


Ablate 12 attention heads in layer 0

In [24]:
to_ablate = int(model.config.num_attention_heads*0.75)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=False,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198, 151667,    271, 151668,    271])
Output tokens: tensor([ 33263, 125582,  99485,   8908, 129556,  99777,    238, 146510,  17796,
         13157,  31431,  45088,  63877,  32820,    509,    292,  47638,  34603,
         36214,  17569,  64705,   5305,  28458,    309,   4330,    329,   6357,
           266,    349,  29812,  73476,    505,    327,  39441,    632,  10187,
          1608,  66103,  21334,   2268,   3795,   1448,     12,    220,    609,
           279,    855,    825,    323,    259])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] in layer 0
user
The capital of France is
assistant
<think>

</think>

−ợi� � tố縐ប oc vit vicoviogiiviociciccacciadiciacaacaaaaaaamatoaditoatate ate ata exexexcite-exited-ext-pre-p-r-m-  & the pre one and t


Ablate 16 (all) attention heads in layer 0

In [25]:
to_ablate = int(model.config.num_attention_heads)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.5,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Output tokens: tensor([ 31305,   7552, 124192,   4838,    220,     13,  44022,   6949, 131069,
           568,  45934,  52798,    353,  19922,      9, 145264, 144540,  47096,
         13282,  11665,  13509,   7276,    968,  71953,   3202,  37572,   8835,
            11,  41873,     25,  29762,  64281,  71807,   1112,    271,    659,
          3131,   2146,  24137,  71179,  33450,  10002,   1090, 147007,  59108,
        145145, 141896,  43701,  32795,  86975])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] in layer 0
user
The capital of France is
assistant
�）べsemb .ёдён).=.� **.*∧≈æk Foxxi Kim-FngÀãïâ,¼:ι ÀÎ...

 .ç...

îỉể::$igh╔ ÎǐỂÁÂÄ


Ablate all attention heads in layer 11

In [26]:
to_ablate = int(model.config.num_attention_heads)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=11,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=False,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198, 151667,    271, 151668,    271])
Outputs: tensor([   785,   6722,    315,   9625,    374,  12095,     13, 151645])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] in layer 11
user
The capital of France is
assistant
<think>

</think>

The capital of France is Paris.


Ablate heads in every layer

Baseline

In [27]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Baseline")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Baseline
user
The capital of France is
assistant
<think>
Okay, the user asked, "The capital of France is." I need to provide the correct answer. Let me think.

First, I know that France's capital is Paris. But wait, maybe they want more details? Like why it


In [28]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [0] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed head 0 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed head 0 in all layers
user
The capital of France is
assistant
<think>
Okay, the user asked for the capital of France. I need to make sure I get this right.

First, I remember that Paris is the capital of France. But wait, let me double-check. Yes, France's capital is definitely


In [29]:
to_ablate = int(model.config.num_attention_heads*0.25)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3 in all layers
user
The capital of France is
assistant
:
ا الصوابكيب the.? No, we have to... Wait! I'm sorry for that question:

 there's a no one day ago.

1 ( every time when then now here are you say "I see your saying so


In [30]:
to_ablate = int(model.config.num_attention_heads*0.5)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3,4,5,6,7 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3,4,5,6,7 in all layers
user
The capital of France is
assistant
1: 2ne,ncing'sraseb. scenert's

nces surl naser on neigerish en rhil gos; in shgas tOY9 eh gstanserics.

med=g


In [31]:
to_ablate = int(model.config.num_attention_heads)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Output tokens: tensor([  1170,     35,  52704,  39524,  57991,  11538,   1154,     13,  26976,
         51108,  86696,   3606,  97525,  31602, 139723,  68741,  74307, 113401,
           237, 113243,  99278,    671,  27393,    900,     71,  21651,   2329,
          1014,   1002,    821,   8579,   6469,  62410,   4758,  73022,  86615,
         28017,  32186,     61,  10606, 125115,    224,  78750, 109806,  44793,
         28849,  35826,   4275,  84005, 100034])

Zeroed all heads in all layers
user
The capital of France is
assistant
lectionDِ :(pto Case ,.inde iff/poweramedfried ^=rawidłow Bj skeletons我院�经验和带 #zelnderh£ lawmlitle datadiaming'clockirectory LeighairportLECTION _$^amiễn�.argument日照道#elif borderColorelvesverbatim经验


In [32]:
to_ablate = int(model.config.num_attention_heads)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Output tokens: tensor([  1170,     35,  52704,  39524,  57991,  11538,   1154,     13,  26976,
         51108,  86696,   3606,  97525,  31602, 139723,  68741,  74307, 113401,
           237, 113243,  99278,    671,  27393,    900,     71,  21651,   2329,
          1014,   1002,    821,   8579,   6469,  62410,   4758,  73022,  86615,
         28017,  32186,     61,  10606, 125115,    224,  78750, 109806,  44793,
         28849,  35826,   4275,  84005, 100034])

Zeroed all heads in all layers
user
The capital of France is
assistant
lectionDِ :(pto Case ,.inde iff/poweramedfried ^=rawidłow Bj skeletons我院�经验和带 #zelnderh£ lawmlitle datadiaming'clockirectory LeighairportLECTION _$^amiễn�.argument日照道#elif borderColorelvesverbatim经验
